# 01. Data Audit

This notebook performs an initial audit of the financial transaction dataset.

The aim is to understand the structure, quality and limitations of the data before
carrying out exploratory analysis or building machine-learning models.

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Fraud Detection - Data Audit")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(spark.version)

4.1.1


In [3]:
data_path = "data/financial_fraud_detection_dataset.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(data_path)
)

print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

Rows: 5,000,000
Columns: 18


In [4]:
df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- sender_account: string (nullable = true)
 |-- receiver_account: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- location: string (nullable = true)
 |-- device_used: string (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- fraud_type: string (nullable = true)
 |-- time_since_last_transaction: double (nullable = true)
 |-- spending_deviation_score: double (nullable = true)
 |-- velocity_score: integer (nullable = true)
 |-- geo_anomaly_score: double (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- device_hash: string (nullable = true)



In [5]:
df.columns

['transaction_id',
 'timestamp',
 'sender_account',
 'receiver_account',
 'amount',
 'transaction_type',
 'merchant_category',
 'location',
 'device_used',
 'is_fraud',
 'fraud_type',
 'time_since_last_transaction',
 'spending_deviation_score',
 'velocity_score',
 'geo_anomaly_score',
 'payment_channel',
 'ip_address',
 'device_hash']

In [6]:
df.show(5, truncate=False)

+--------------+--------------------------+--------------+----------------+-------+----------------+-----------------+--------+-----------+--------+----------+---------------------------+------------------------+--------------+-----------------+---------------+---------------+-----------+
|transaction_id|timestamp                 |sender_account|receiver_account|amount |transaction_type|merchant_category|location|device_used|is_fraud|fraud_type|time_since_last_transaction|spending_deviation_score|velocity_score|geo_anomaly_score|payment_channel|ip_address     |device_hash|
+--------------+--------------------------+--------------+----------------+-------+----------------+-----------------+--------+-----------+--------+----------+---------------------------+------------------------+--------------+-----------------+---------------+---------------+-----------+
|T100000       |2023-08-22 09:22:43.516168|ACC877572     |ACC388389       |343.78 |withdrawal      |utilities        |Tokyo   |mob

In [7]:
df.head(1)

[Row(transaction_id='T100000', timestamp=datetime.datetime(2023, 8, 22, 9, 22, 43, 516168), sender_account='ACC877572', receiver_account='ACC388389', amount=343.78, transaction_type='withdrawal', merchant_category='utilities', location='Tokyo', device_used='mobile', is_fraud=False, fraud_type=None, time_since_last_transaction=None, spending_deviation_score=-0.21, velocity_score=3, geo_anomaly_score=0.22, payment_channel='card', ip_address='13.101.214.112', device_hash='D8536477')]

In [8]:
from pyspark.sql.functions import col, sum, when

missing = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

missing.show()

+--------------+---------+--------------+----------------+------+----------------+-----------------+--------+-----------+--------+----------+---------------------------+------------------------+--------------+-----------------+---------------+----------+-----------+
|transaction_id|timestamp|sender_account|receiver_account|amount|transaction_type|merchant_category|location|device_used|is_fraud|fraud_type|time_since_last_transaction|spending_deviation_score|velocity_score|geo_anomaly_score|payment_channel|ip_address|device_hash|
+--------------+---------+--------------+----------------+------+----------------+-----------------+--------+-----------+--------+----------+---------------------------+------------------------+--------------+-----------------+---------------+----------+-----------+
|             0|        0|             0|               0|     0|               0|                0|       0|          0|       0|   4820447|                     896513|                       0|     

In [9]:
missing_counts = (
    df.select([
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])
    .collect()[0]
)

missing_summary = [
    (column, missing_counts[column])
    for column in df.columns
]

spark.createDataFrame(
    missing_summary,
    ["column", "missing_count"]
).orderBy(col("missing_count").desc()).show()

+--------------------+-------------+
|              column|missing_count|
+--------------------+-------------+
|          fraud_type|      4820447|
|time_since_last_t...|       896513|
|          ip_address|            0|
|      transaction_id|            0|
|         device_hash|            0|
|spending_deviatio...|            0|
|            location|            0|
|           timestamp|            0|
|     payment_channel|            0|
|      sender_account|            0|
|              amount|            0|
|   merchant_category|            0|
|    transaction_type|            0|
|    receiver_account|            0|
|            is_fraud|            0|
|      velocity_score|            0|
|         device_used|            0|
|   geo_anomaly_score|            0|
+--------------------+-------------+



In [10]:
total    = df.count()                                      #total rows
distinct = df.dropDuplicates(["transaction_id"]).count()   #unique rows
duplicates = total - distinct                              #difference = duplicates

print(f"Total rows:      {total:,}")
print(f"Distinct rows:   {distinct:,}")
print(f"Duplicate rows:  {duplicates:,}")

Total rows:      5,000,000
Distinct rows:   5,000,000
Duplicate rows:  0


In [11]:
from pyspark.sql.functions import count

df.groupBy("transaction_id") \
    .agg(count("*").alias("count")) \
    .filter(col("count") > 1) \
    .count()

0

In [12]:
cardinality = []

for c in df.columns:
    cardinality.append(
        (c, df.select(c).distinct().count())
    )

spark.createDataFrame(
    cardinality,
    ["column", "unique_values"]
).show(truncate=False)

+---------------------------+-------------+
|column                     |unique_values|
+---------------------------+-------------+
|transaction_id             |5000000      |
|timestamp                  |4999998      |
|sender_account             |896513       |
|receiver_account           |896639       |
|amount                     |217069       |
|transaction_type           |4            |
|merchant_category          |8            |
|location                   |8            |
|device_used                |4            |
|is_fraud                   |2            |
|fraud_type                 |2            |
|time_since_last_transaction|4103488      |
|spending_deviation_score   |917          |
|velocity_score             |20           |
|geo_anomaly_score          |101          |
|payment_channel            |4            |
|ip_address                 |4997068      |
|device_hash                |3835723      |
+---------------------------+-------------+



In [11]:
df.groupBy("is_fraud") \
    .count() \
    .orderBy("is_fraud") \
    .show()

+--------+-------+
|is_fraud|  count|
+--------+-------+
|   false|4820447|
|    true| 179553|
+--------+-------+



In [15]:
from pyspark.sql.functions import round

target_distribution = (
    df.groupBy("is_fraud")
      .count()
      .withColumn(
          "percentage",
          round(col("count") / total * 100, 3)
      )
      .orderBy("is_fraud")
)

target_distribution.show()

+--------+-------+----------+
|is_fraud|  count|percentage|
+--------+-------+----------+
|   false|4820447|    96.409|
|    true| 179553|     3.591|
+--------+-------+----------+



In [16]:
df.select("amount").summary().show()

+-------+------------------+
|summary|            amount|
+-------+------------------+
|  count|           5000000|
|   mean| 358.9342687640248|
| stddev|469.93331106593763|
|    min|              0.01|
|    25%|             26.57|
|    50%|            138.66|
|    75%|            503.84|
|    max|           3520.57|
+-------+------------------+



In [17]:
df.select(
    "amount",
    "time_since_last_transaction",
    "spending_deviation_score",
    "velocity_score",
    "geo_anomaly_score"
).summary().show()

26/09/21 10:51:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+---------------------------+------------------------+-----------------+-------------------+
|summary|            amount|time_since_last_transaction|spending_deviation_score|   velocity_score|  geo_anomaly_score|
+-------+------------------+---------------------------+------------------------+-----------------+-------------------+
|  count|           5000000|                    4103487|                 5000000|          5000000|            5000000|
|   mean| 358.9342687640248|         1.5257988852641178|    -3.88115999999991...|       10.5013196| 0.5000292560000076|
| stddev|469.93331106593763|           3576.56870189105|      1.0008069793965741|5.766842438453652|0.28863493504098214|
|    min|              0.01|         -8777.814181944444|                   -5.26|                1|                0.0|
|    25%|             26.57|         -2562.326092625278|                   -0.68|                6|               0.25|
|    50%|            138.66|         0.8

In [16]:
df.select(col("time_since_last_transaction").isNotNull()).select("time_since_last_transaction").show(20, truncate=False)

{"ts": "2026-10-05 10:27:26.819", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `time_since_last_transaction` cannot be resolved. Did you mean one of the following? [`(time_since_last_transaction IS NOT NULL)`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor50.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o590.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `time_since_last_transaction` cannot be resolved. Did you mean one of the following? [`(time_since_last_transaction IS NOT NULL)`]. SQLSTATE: 42703;\n'Project ['time_since_last_transaction]\n+- Project [isnotnull(time_since_last_transaction#28) AS (time_since_l

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `time_since_last_transaction` cannot be resolved. Did you mean one of the following? [`(time_since_last_transaction IS NOT NULL)`]. SQLSTATE: 42703;
'Project ['time_since_last_transaction]
+- Project [isnotnull(time_since_last_transaction#28) AS (time_since_last_transaction IS NOT NULL)#696]
   +- Relation [transaction_id#17,timestamp#18,sender_account#19,receiver_account#20,amount#21,transaction_type#22,merchant_category#23,location#24,device_used#25,is_fraud#26,fraud_type#27,time_since_last_transaction#28,spending_deviation_score#29,velocity_score#30,geo_anomaly_score#31,payment_channel#32,ip_address#33,device_hash#34] csv


In [18]:
df.select("amount").filter(col("amount") < 0).count()

0

In [19]:
df.select("spending_deviation_score").filter(
    col("spending_deviation_score") < 0
).count()

2490173

26/09/21 11:19:54 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1244545 ms exceeds timeout 120000 ms
26/09/21 11:19:57 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/21 11:20:00 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

## Potential target leakage

The variable `fraud_type` is associated directly with the fraud outcome and would
not be available as an independent predictor when making a real-time fraud
decision. It is therefore identified as a potential source of target leakage
and will be excluded from the modelling dataset.

Other identifier variables, such as transaction IDs and IP addresses, are also
reviewed separately because their high cardinality may limit their usefulness
as direct model features.